# TTD Dataset Demo

This notebook follows the modality guidance called out in issue #126 and demonstrates:
- Direct SQL
- REST API
- MCP tools
- Agentic access through dataset `/rpc` (orchestrator optional)


In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
import requests

REQUEST_TIMEOUT = int(os.getenv("TTD_NOTEBOOK_TIMEOUT", "20"))
AGENTIC_TIMEOUT = int(os.getenv("TTD_AGENTIC_TIMEOUT", "90"))
TTD_NODE_URL = os.getenv("TTD_NODE_URL", "http://ca-ttd-agentic-prod").rstrip("/")
ORCHESTRATOR_URL = os.getenv(
    "ORCHESTRATOR_URL",
    "https://ca-orchestrator-agentic-prod.bluesmoke-f9a71a0e.westus3.azurecontainerapps.io",
).rstrip("/")
JUPYTER_NODE_URL = os.getenv("JUPYTER_NODE_URL", "http://localhost:8002").rstrip("/")
ORCHESTRATOR_SESSION_TOKEN = os.getenv("ORCHESTRATOR_SESSION_TOKEN", "").strip()
TTD_DB_PATH = Path(os.getenv("TTD_DB_PATH", "/app/data/ttd.sqlite3"))

if not TTD_NODE_URL:
    raise RuntimeError("TTD_NODE_URL is required but empty. Set TTD_NODE_URL and rerun.")

def _build_url(path: str) -> str:
    if path.startswith("http://") or path.startswith("https://"):
        return path
    return f"{TTD_NODE_URL}{path}"

def api_get(path: str, **params):
    return requests.get(_build_url(path), params=params or None, timeout=REQUEST_TIMEOUT)

def api_post(path: str, payload: dict):
    return requests.post(_build_url(path), json=payload, timeout=REQUEST_TIMEOUT)

def assert_ok(response, context: str):
    if response.status_code != 200:
        snippet = response.text[:500]
        raise RuntimeError(f"{context} failed: status={response.status_code}, body={snippet}")

def preview_records(records, n=5):
    rows = records[:n] if isinstance(records, list) else []
    if not rows:
        print("No records to preview.")
        return
    print(json.dumps(rows, indent=2)[:2000])

print("TTD_NODE_URL:", TTD_NODE_URL)
print("ORCHESTRATOR_URL:", ORCHESTRATOR_URL)
print("JUPYTER_NODE_URL:", JUPYTER_NODE_URL)
print("ORCHESTRATOR_SESSION_TOKEN set:", bool(ORCHESTRATOR_SESSION_TOKEN))
print("TTD_DB_PATH:", TTD_DB_PATH)
print("pandas version:", pd.__version__)
print("agentic timeout:", AGENTIC_TIMEOUT)


## Section A: Setup and environment preflight
Validate endpoint reachability before running modality-specific checks.


In [ ]:
checks = [
    ("ttd-health", "/health"),
    ("ttd-ready", "/ready"),
    ("ttd-summary", "/data/summary"),
]

for name, path in checks:
    try:
        resp = api_get(path)
        print(f"{name:20s} -> {resp.status_code}")
        assert_ok(resp, name)
    except Exception as exc:
        print(f"{name:20s} -> ERROR: {exc}")

# Optional orchestrator visibility check (helpful when running end-to-end stack).
try:
    orch_resp = requests.get(f"{ORCHESTRATOR_URL}/health", timeout=REQUEST_TIMEOUT)
    print(f"{'orchestrator-health':20s} -> {orch_resp.status_code}")
except Exception as exc:
    print(f"{'orchestrator-health':20s} -> ERROR: {exc}")


## Section B: Direct SQL
This section demonstrates SQL-level access in two ways:
1. Direct local SQLite file access (if DB file is available in notebook runtime)
2. SQL submitted through the node `/data/query` route (read-only guard enforced by node)


In [ ]:
# Count-first diagnostics keep this section actionable when a deployed node is healthy but unseeded.
count_sql = "SELECT COUNT(*) AS interactions_count FROM interactions"

direct_sql_query = """
SELECT c.name, t.target_name, i.metric_type, i.metric_value, i.metric_unit
FROM interactions i
JOIN compounds c ON c.id = i.compound_id
JOIN targets t ON t.id = i.target_id
LIMIT 25
"""

print("=== B0) Row count diagnostic ===")
row_count = None
try:
    resp = requests.post(
        f"{TTD_NODE_URL}/data/query",
        json={"sql": count_sql},
        timeout=REQUEST_TIMEOUT,
    )
    payload = resp.json()
    if resp.status_code == 200 and payload.get("rows"):
        row_count = payload["rows"][0].get("interactions_count")
    print("status:", resp.status_code, "interactions_count:", row_count)
except Exception as exc:
    print("count diagnostic error:", exc)

print("\n=== B1) Local SQLite direct SQL ===")
if TTD_DB_PATH.exists():
    with sqlite3.connect(TTD_DB_PATH) as conn:
        rows = conn.execute(direct_sql_query).fetchall()
    print(f"rows={len(rows)}")
    for row in rows[:5]:
        print(row)
else:
    print(f"Local DB not found at {TTD_DB_PATH}; skipping direct-file SQL check.")

print("\n=== B2) SQL via /data/query -> pandas DataFrame ===")
try:
    resp = requests.post(
        f"{TTD_NODE_URL}/data/query",
        json={"sql": direct_sql_query},
        timeout=REQUEST_TIMEOUT,
    )
    print("status:", resp.status_code)
    payload = resp.json()
    rows = payload.get("rows", [])
    df_sql = pd.DataFrame(rows)
    print("row_count:", payload.get("count", len(df_sql)))
    if not df_sql.empty:
        print(df_sql.head(10).to_string(index=False))
    else:
        print("NOTE: Query succeeded but returned zero rows. This usually means the TTD dataset is not seeded in this environment yet.")
        print("Run Section C (/data/summary and /data/tables) to confirm row counts.")
except Exception as exc:
    print("query error:", exc)


### B3: Explicit pandas DataFrame example

This example is intentionally pandas-first: query with `/data/query`, load into a DataFrame, and render tabular output directly.


In [ ]:
pandas_sql = """
SELECT c.name, t.target_name, i.metric_type, i.metric_value, i.metric_unit
FROM interactions i
JOIN compounds c ON c.id = i.compound_id
JOIN targets t ON t.id = i.target_id
ORDER BY i.metric_value ASC
LIMIT 10
"""

try:
    resp = api_post("/data/query", {"sql": pandas_sql})
    print("status:", resp.status_code)
    assert_ok(resp, "pandas /data/query")

    payload = resp.json()
    df_pandas = pd.DataFrame(payload.get("rows", []))
    print("row_count:", payload.get("count", len(df_pandas)))

    if df_pandas.empty:
        print("No rows returned. Dataset may be unseeded in this environment.")
    else:
        df_pandas.head(10)
except Exception as exc:
    print("pandas example error:", exc)


## Section C: REST API
Call dataset-specific and generic data routes directly on the TTD node.


In [ ]:
print("summary:")
try:
    r = api_get("/data/summary")
    print("status:", r.status_code)
    body = r.json()
    print(json.dumps(body, indent=2)[:1200])
except Exception as exc:
    print("summary -> ERROR:", exc)

print("\ntables:")
try:
    r = api_get("/data/tables")
    print("status:", r.status_code)
    body = r.json()
    tables = body.get("tables", []) if isinstance(body, dict) else []
    print("table_count:", len(tables))
    preview_records(tables, n=5)
except Exception as exc:
    print("tables -> ERROR:", exc)

print("\nttd-search-egfr:")
try:
    r = api_get("/ttd/search", q="EGFR", limit=5)
    print("status:", r.status_code)
    body = r.json()
    rows = body.get("rows", []) if isinstance(body, dict) else []
    print("row_count:", body.get("count", len(rows)) if isinstance(body, dict) else len(rows))
    preview_records(rows, n=5)
except Exception as exc:
    print("ttd-search-egfr -> ERROR:", exc)

print("\nttd-target-p00533:")
try:
    r = api_get("/ttd/target/P00533")
    print("status:", r.status_code)
    body = r.json()
    print(json.dumps(body, indent=2)[:1200])
except Exception as exc:
    print("ttd-target-p00533 -> ERROR:", exc)


## Section D: MCP Tools
Discover MCP tools from the TTD node and invoke a tool call.


In [ ]:
mcp_tools = []
try:
    r = requests.get(f"{TTD_NODE_URL}/mcp/tools", timeout=REQUEST_TIMEOUT)
    print("/mcp/tools status:", r.status_code)
    data = r.json()
    mcp_tools = data if isinstance(data, list) else data.get("tools", [])
    print("tool_count:", len(mcp_tools))
    for t in mcp_tools[:10]:
        name = t.get("name") if isinstance(t, dict) else None
        desc = t.get("description") if isinstance(t, dict) else None
        print("-", name, "::", (desc or "")[:120])
except Exception as exc:
    print("mcp tools discovery error:", exc)


In [ ]:
candidate_names = []
for t in mcp_tools:
    if not isinstance(t, dict):
        continue
    nm = (t.get("name") or "").strip()
    if nm:
        candidate_names.append(nm)

search_tool = next((n for n in candidate_names if "search" in n.lower()), None)
target_tool = next((n for n in candidate_names if "target" in n.lower()), None)
tool_name = search_tool or target_tool

print("selected_tool:", tool_name)
if not tool_name:
    print("No usable MCP tool discovered in /mcp/tools output.")
else:
    is_search = "search" in tool_name.lower()
    # Use P00533 so seeded baseline data returns non-empty rows.
    search_args = {"query": "P00533", "limit": 5}
    target_args = {"uniprot_id": "P00533"}

    payload_new = {
        "name": tool_name,
        "arguments": search_args if is_search else target_args,
    }
    payload_legacy = {
        "tool": tool_name,
        "args": search_args if is_search else target_args,
    }
    payload_q_fallback = {
        "name": tool_name,
        "arguments": {"q": "P00533", "limit": 5} if is_search else target_args,
    }

    try:
        resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_new, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400:
            resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_legacy, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400 and is_search:
            resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_q_fallback, timeout=REQUEST_TIMEOUT)

        print("/mcp/call status:", resp.status_code)
        try:
            body = resp.json()
            rows = body.get("rows", []) if isinstance(body, dict) else []
            print("row_count:", body.get("count") if isinstance(body, dict) else "n/a")
            print(json.dumps(rows[:3], indent=2) if rows else resp.text[:1200])
        except Exception:
            print(resp.text[:1200])
    except Exception as exc:
        print("mcp call error:", exc)


## Section E: Agentic Access (RPC-first)
Use dataset-node `/rpc` as the primary notebook path. Print the actual agent response so users can validate behavior directly, then provide an optional concrete fallback summary when needed.


In [ ]:
query = "Find high-affinity EGFR interactions from the TTD dataset and summarize top hits."

print("sending to:", f"{TTD_NODE_URL}/rpc")
print("query:", query)

def _local_summary_fallback() -> None:
    print("\nOptional concrete fallback summary from /ttd/search (for comparison):")
    try:
        r = requests.get(
            f"{TTD_NODE_URL}/ttd/search",
            params={"q": "P00533", "limit": 5},
            timeout=REQUEST_TIMEOUT,
        )
        print("fallback status:", r.status_code)
        body = r.json()
        rows = body.get("rows", []) if isinstance(body, dict) else []
        if not rows:
            print("No rows found for fallback summary.")
            return

        for row in rows[:5]:
            print(
                f"- {row.get('compound')} vs {row.get('target_name')} "
                f"({row.get('uniprot_id')}): {row.get('metric_type')}={row.get('metric_value')} {row.get('metric_unit')}"
            )
    except Exception as exc:
        print("fallback failed:", exc)

def _extract_assistant_text(body: dict) -> str:
    if not isinstance(body, dict):
        return ""
    parts = ((body.get("result") or {}).get("parts") or [])
    chunks = [p.get("text", "") for p in parts if isinstance(p, dict) and p.get("kind") == "text"]
    return "\n".join([c for c in chunks if c]).strip()

rpc_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "kind": "message",
            "messageId": "ttd-demo-agentic-1",
            "parts": [{"kind": "text", "text": query}],
        },
        "metadata": {},
    },
}

def _call_rpc_once(label: str):
    resp = requests.post(
        f"{TTD_NODE_URL}/rpc",
        json=rpc_payload,
        timeout=AGENTIC_TIMEOUT,
    )
    print(f"{label} status:", resp.status_code)
    resp.raise_for_status()
    body = resp.json()
    answer = _extract_assistant_text(body)

    print("\nassistant response (raw):\n")
    if answer:
        print(answer[:4000])
    else:
        print("<no text parts returned by agent>")

    return answer

try:
    answer = _call_rpc_once("status")
except Exception as exc:
    print("agentic RPC call failed:", exc)
    print("Retrying once with the same payload...")
    try:
        answer = _call_rpc_once("retry")
    except Exception as exc2:
        print("retry failed:", exc2)
        answer = ""

# Keep fallback as an optional confidence boost, but never replace agent output text.
if not answer or "local BindingDB + TTD scaffold dataset" in answer:
    _local_summary_fallback()


## Section F: Troubleshooting and rerun guide

Use this quick map when a section fails:

1. 404 route mismatch
- Verify `TTD_NODE_URL` points to the TTD dataset node and does not include an extra path prefix.
- Re-run Section A and confirm `/health`, `/ready`, and `/data/summary` all return 200.

2. 5xx or timeout
- The service may be starting or under load. Re-run the failed cell once after 10-20 seconds.
- If Section E times out, increase `TTD_AGENTIC_TIMEOUT` and rerun the setup cell.

3. Env/config mismatch
- Confirm `TTD_NODE_URL` is reachable from the notebook runtime.
- If orchestrator checks fail but TTD checks pass, continue with dataset-direct paths (`/data/*`, `/mcp/*`, `/rpc`).

4. Empty dataset behavior
- A 200 response with zero rows usually indicates unseeded data in the current environment.
- Use Section B row-count diagnostic and Section C summary/tables output to confirm data state.
